# COVID-19 Clinical Trials — Exploratory Data Analysis

Exploratory analysis of 5,783 COVID-19 clinical trials registered on ClinicalTrials.gov, covering trial status, phase, study type, funding source, enrollment size, and how the target condition is labeled across trials.

**Dataset:** `COVID clinical trials - COVID clinical trials.csv` (this repo)

**Questions this notebook answers:**
1. What state are these trials in (recruiting, completed, withdrawn, etc.)?
2. What phase of testing are they at, and what study type (interventional vs. observational)?
3. Who funds COVID-19 trials — industry, government, or other?
4. How large are these trials, and how skewed is enrollment?
5. How consistently is the condition itself labeled across 5,783 independently-submitted trials?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv('COVID clinical trials - COVID clinical trials.csv')
print(df.shape)
df.head()

## 1. Data overview and missingness

In [ ]:
df.info()

missing = df.isna().mean().sort_values(ascending=False) * 100
missing[missing > 0].round(1)

## 2. Trial status

Nearly half of all registered trials are still recruiting, which makes sense for a dataset pulled during an active pandemic — this is a live registry snapshot, not a closed retrospective dataset.

In [ ]:
status_counts = df['Status'].value_counts()
print(status_counts)

plt.figure(figsize=(9, 5))
sns.barplot(x=status_counts.values, y=status_counts.index, hue=status_counts.index, palette='viridis', legend=False)
plt.title('COVID-19 Clinical Trials by Status')
plt.xlabel('Number of Trials')
plt.tight_layout()
plt.show()

## 3. Study phase and study type

"Not Applicable" is the single largest phase category — expected, since a large share of COVID-19 research (diagnostics, observational cohort studies, device trials) doesn't go through the drug-trial phase system at all. Interventional trials outnumber observational ones roughly 3:2.

In [ ]:
phase_counts = df['Phases'].value_counts().head(10)
study_type_counts = df['Study Type'].value_counts().head(6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=phase_counts.values, y=phase_counts.index, hue=phase_counts.index, ax=axes[0], palette='mako', legend=False)
axes[0].set_title('Trials by Phase')
sns.barplot(x=study_type_counts.values, y=study_type_counts.index, hue=study_type_counts.index, ax=axes[1], palette='mako', legend=False)
axes[1].set_title('Trials by Study Type')
plt.tight_layout()
plt.show()

## 4. Who funds COVID-19 trials?

In [ ]:
funded_counts = df['Funded Bys'].value_counts().head(8)
print(funded_counts)

plt.figure(figsize=(8, 5))
sns.barplot(x=funded_counts.values, y=funded_counts.index, hue=funded_counts.index, palette='crest', legend=False)
plt.title('Trials by Funding Source')
plt.xlabel('Number of Trials')
plt.tight_layout()
plt.show()

The large majority of trials (~78%) are funded by "Other" (typically academic/hospital sponsors) rather than industry or government — consistent with COVID-19 research being driven heavily by academic medical centers responding rapidly to an emerging crisis, rather than industry-led drug development.

## 5. Enrollment size — and why the mean is misleading

In [ ]:
enrollment = df['Enrollment'].dropna()
print(f"n = {len(enrollment)}")
print(f"mean     = {enrollment.mean():,.0f}")
print(f"median   = {enrollment.median():,.0f}")
print(f"max      = {enrollment.max():,.0f}")
print(f"95th pct = {enrollment.quantile(0.95):,.0f}")

The mean enrollment (~18,300) is wildly skewed by a small number of massive registry-scale studies (the largest exceeds 20 million participants). The **median (170)** is a far more honest description of a typical COVID-19 trial — this is a textbook case for reporting median over mean on enrollment-size data, and worth calling out explicitly rather than quoting the mean at face value.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(enrollment[enrollment < enrollment.quantile(0.95)], bins=40, kde=True)
plt.title('Enrollment Distribution (below 95th percentile, to exclude extreme outliers)')
plt.xlabel('Enrollment')
plt.tight_layout()
plt.show()

## 6. Condition labeling — a data quality finding

This dataset is self-reported by thousands of independent trial sponsors, and it shows: the *same* disease appears under at least ten different spellings/labels in the `Conditions` field.

In [ ]:
conditions = df['Conditions'].dropna().str.split('|').explode().str.strip()
top_conditions = conditions.value_counts().head(10)
top_conditions

Ten of the top labels — `Covid19`, `COVID-19`, `COVID`, `Covid-19`, `Corona Virus Infection`, `Coronavirus`, `Coronavirus Infection`, `SARS-CoV-2`, `SARS-CoV Infection`, `SARS-CoV 2` — all refer to the same underlying condition. Naively grouping by `Conditions` as-is would badly undercount how many trials actually study COVID-19. A real downstream analysis would need a normalization/mapping step (e.g. a regex or lookup table collapsing these variants) before condition-level aggregation is trustworthy — this is flagged here rather than silently worked around, since it's the kind of data-quality issue that's easy to miss and would quietly bias any condition-level conclusions.

## Summary

- **5,783** COVID-19 trials analyzed; **~48%** still recruiting at time of data pull.
- Interventional trials outnumber observational ones (3,322 vs. 2,427); "Not Applicable" is the largest single phase category, reflecting the mix of non-drug study designs in the dataset.
- **~78%** of trials are funded by non-industry, non-government ("Other") sponsors — mostly academic and hospital-led research.
- Enrollment is heavily right-skewed (mean ~18,300 vs. median 170); median is the honest summary statistic here.
- The `Conditions` field has a significant label-consistency problem — at least 10 distinct strings for the same disease — that would need normalization before any condition-based aggregation could be trusted.

## What I'd Add Next
- A normalization step for `Conditions` and a re-run of condition-level counts
- A timeline view of trial start dates to show enrollment ramp-up over the pandemic
- Geographic breakdown using the `Locations` field